# IPM data: load & troubleshoot

Use this notebook to **inspect IPM variables** in the saved joint dataset, **smoke-test** the National IPM Database API, and **optionally rebuild** document-level tables without running the full `build_joint_dataset.ipynb` pipeline.

| Resource | Location |
|----------|----------|
| Schema & interpretation | [`docs/schema_ipm_integration.md`](../docs/schema_ipm_integration.md) |
| Full pipeline (CDL + IPM + PLACES + save) | [`build_joint_dataset.ipynb`](build_joint_dataset.ipynb) |
| Implementation | [`ipm_pipeline.py`](ipm_pipeline.py) |

**Typical workflow:** run **Part 1** (offline) on the joint CSV → if numbers look wrong, run **Part 2** (needs network) → for deep dives, run **Part 3** (slow if PDF/HTML enabled).

## Part 1 — IPM columns in the saved joint dataset (offline)

Loads `data/joint_county_year_2018_2019.csv` (or set `JOINT_CSV` below). No API calls.

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd

REPO_ROOT = Path("..").resolve()
JOINT_CSV = REPO_ROOT / "data" / "joint_county_year_2018_2019.csv"

assert JOINT_CSV.exists(), f"Missing {JOINT_CSV} — run build_joint_dataset.ipynb first."
joint = pd.read_csv(JOINT_CSV, low_memory=False)
print("Joint shape:", joint.shape)

IPM_KNOWN = [
    "ipm_breadth_acre", "ipm_breadth_acre_rescaled", "ipm_breadth_value",
    "chemical_reliance_acre", "chemical_reliance_acre_rescaled", "chemical_reliance_value",
    "ipm_doc_coverage_share", "mean_text_quality", "mean_geo_confidence", "weighted_doc_age",
    "ipm_primary_match_tier", "county_crop_concentration", "specialty_crop_share", "total_ag_value",
]
ipm_cols = [c for c in IPM_KNOWN if c in joint.columns]
ipm_cols += [c for c in joint.columns if c.startswith("ipm_") and c not in ipm_cols]
ipm_cols += [c for c in joint.columns if c.startswith("chemical_reliance") and c not in ipm_cols]
ipm_cols = list(dict.fromkeys(ipm_cols))
print("IPM-related columns:", ipm_cols)

j = joint[ipm_cols].copy()
display(j.describe().T)

print("\n--- Missingness (share non-null) ---")
miss = j.notna().mean().sort_values()
display(miss.to_frame("non_null_share"))

# --- Auto-diagnosis (your CSV pattern: comp OK, IPM all null, coverage 0) ---
_has_comp = "county_crop_concentration" in joint.columns and joint["county_crop_concentration"].notna().any()
_has_ipm = "ipm_breadth_acre" in joint.columns and joint["ipm_breadth_acre"].notna().any()
_cov_any = ("ipm_doc_coverage_share" in joint.columns) and (joint["ipm_doc_coverage_share"] > 0).any()
if _has_comp and not _has_ipm and not _cov_any:
    print("\n" + "=" * 62)
    print("DIAGNOSIS")
    print("=" * 62)
    print(
        "IPM scores are 100% missing and ipm_doc_coverage_share is 0 everywhere,\n"
        "but county_crop_concentration has data. That means:\n"
        "  • County–year crop composition (CDL) merged into the joint file OK.\n"
        "  • No county–crop–year row ever got an IPM document match — so when\n"
        "    the CSV was built, crop_geo_doc_ipm was empty OR the API step failed\n"
        "    (offline run, timeout, or exception → see build_joint §3b console).\n"
    )
    print("What to do:")
    print("  1. Run Part 2 below — if API fails, fix network/VPN and retry.")
    print("  2. Re-run build_joint_dataset.ipynb from §3b through save, with LIGHT_RUN=False")
    print("     and stable network; confirm you do NOT see 'build_crop_geo_doc_ipm failed'.")
    print("  3. Optional: Part 3 with USE_PDF_FALLBACK=False should return hundreds of")
    print("     crop×state doc rows in seconds; if shape is (0, n), debug API there.")
    print("=" * 62 + "\n")
elif _has_ipm:
    print("\n(At least some rows have non-null ipm_breadth_acre — IPM merge ran.)\n")

year_col = "YEAR" if "YEAR" in joint.columns else ("year" if "year" in joint.columns else None)
if year_col:
    print("\n--- By", year_col, "---")
    for y in sorted(joint[year_col].dropna().unique()):
        sub = joint[joint[year_col] == y]
        if "ipm_breadth_acre" in sub.columns:
            print(f"  {y}: ipm_breadth_acre non-null {sub['ipm_breadth_acre'].notna().mean():.1%}, median {sub['ipm_breadth_acre'].median():.3f}")
        if "ipm_doc_coverage_share" in sub.columns:
            print(f"       ipm_doc_coverage_share > 0: {(sub['ipm_doc_coverage_share'] > 0).mean():.1%}")

state = joint["FIPS"].astype(str).str.zfill(5).str[:2] if "FIPS" in joint.columns else None
if state is not None and "ipm_breadth_acre" in joint.columns:
    print("\n--- States with lowest median ipm_breadth_acre (sample) ---")
    tmp = joint.assign(_st=state)
    med = tmp.groupby("_st")["ipm_breadth_acre"].median().sort_values().head(8)
    display(med)
    print("--- States with highest median ---")
    display(tmp.groupby("_st")["ipm_breadth_acre"].median().sort_values(ascending=False).head(8))

Joint shape: (6128, 488)
IPM-related columns: ['ipm_breadth_acre', 'ipm_breadth_acre_rescaled', 'ipm_breadth_value', 'chemical_reliance_acre', 'chemical_reliance_acre_rescaled', 'chemical_reliance_value', 'ipm_doc_coverage_share', 'mean_text_quality', 'mean_geo_confidence', 'weighted_doc_age', 'ipm_primary_match_tier', 'county_crop_concentration', 'specialty_crop_share', 'total_ag_value']


,count,mean,std,min,25%,50%,75%,max
ipm_breadth_acre,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
ipm_breadth_acre_rescaled,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
ipm_breadth_value,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
chemical_reliance_acre,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
chemical_reliance_acre_rescaled,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
chemical_reliance_value,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
ipm_doc_coverage_share,6114.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
mean_text_quality,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
mean_geo_confidence,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
weighted_doc_age,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN



--- Missingness (share non-null) ---


,non_null_share
ipm_breadth_acre,0.000000
ipm_breadth_acre_rescaled,0.000000
ipm_breadth_value,0.000000
chemical_reliance_acre,0.000000
chemical_reliance_acre_rescaled,0.000000
chemical_reliance_value,0.000000
mean_text_quality,0.000000
mean_geo_confidence,0.000000
weighted_doc_age,0.000000
ipm_primary_match_tier,0.000000



--- By YEAR ---
  2018: ipm_breadth_acre non-null 0.0%, median nan
       ipm_doc_coverage_share > 0: 0.0%
  2019: ipm_breadth_acre non-null 0.0%, median nan
       ipm_doc_coverage_share > 0: 0.0%

--- States with lowest median ipm_breadth_acre (sample) ---


/var/folders/8v/kmqwjstx6sv9d9t9mh2qqpvw0000gn/T/ipykernel_16620/3039866428.py:44: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  tmp = joint.assign(_st=state)


_st
01   NaN
04   NaN
05   NaN
06   NaN
08   NaN
09   NaN
10   NaN
12   NaN
Name: ipm_breadth_acre, dtype: float64

--- States with highest median ---


_st
01   NaN
04   NaN
05   NaN
06   NaN
08   NaN
09   NaN
10   NaN
12   NaN
Name: ipm_breadth_acre, dtype: float64

## Part 2 — National IPM Database API (network)

Quick checks: `/state` and `/source` counts. If this fails, PDF/HTML scoring in Part 3 will also fail.

In [ ]:
import requests
from ipm_pipeline import IPM_API_BASE, REQUEST_TIMEOUT, _ipm_get, _ipm_response_to_rows

def check_api():
    r = requests.get(f"{IPM_API_BASE}/state", timeout=REQUEST_TIMEOUT)
    r.raise_for_status()
    states = _ipm_response_to_rows(r.json())
    print("GET /state OK — rows:", len(states))
    cp = _ipm_response_to_rows(_ipm_get("/source", {"sourcetypeid": 3}))
    pmsp = _ipm_response_to_rows(_ipm_get("/source", {"sourcetypeid": 4}))
    print("Crop Profile sources (id=3):", len(cp))
    print("PMSP sources (id=4):", len(pmsp))
    return states, cp, pmsp

states, cp, pmsp = check_api()
if cp:
    print("Sample Crop Profile keys:", list(cp[0].keys())[:12])

## Part 3 — Rebuild `crop_geo_doc_ipm` (optional, slow)

- **`USE_PDF_FALLBACK=False`** — fast (~seconds): metadata-only scores, good for API/region mapping checks.
- **`USE_PDF_FALLBACK=True`** — fetches PDFs/HTML per document (~5–15+ min).

After running, inspect `crop_geo_doc_ipm` and optionally save to `data/intermediate/crop_geo_doc_ipm.csv`.

In [ ]:
from pathlib import Path

REPO_ROOT = Path("..").resolve()
USE_PDF_FALLBACK = False  # True for full text scoring

from ipm_pipeline import build_crop_geo_doc_ipm

crop_geo_doc_ipm = build_crop_geo_doc_ipm(use_pdf_fallback=USE_PDF_FALLBACK)
print(crop_geo_doc_ipm.shape)
print(crop_geo_doc_ipm["text_source"].value_counts())
print(crop_geo_doc_ipm[["crop", "state_fips", "ipm_breadth_index", "chemical_reliance_index", "text_source"]].head(15))

OUT = REPO_ROOT / "data" / "intermediate"
OUT.mkdir(parents=True, exist_ok=True)
SAVE = False  # set True to write CSV
if SAVE:
    crop_geo_doc_ipm.to_csv(OUT / "crop_geo_doc_ipm.csv", index=False)
    print("Wrote", OUT / "crop_geo_doc_ipm.csv")

## Part 4 — Load saved intermediate (optional)

If you previously exported `crop_geo_doc_ipm.csv`, load it here to avoid re-fetching.

In [ ]:
from pathlib import Path

REPO_ROOT = Path("..").resolve()
path = REPO_ROOT / "data" / "intermediate" / "crop_geo_doc_ipm.csv"
if path.exists():
    crop_geo_doc_ipm = pd.read_csv(path, low_memory=False)
    print("Loaded", path, crop_geo_doc_ipm.shape)
else:
    print("No file at", path, "— run Part 3 with SAVE=True or full joint build.")